# UK Biobank LD across sketch dimensions 

## 1. Generate the projection weight matrix

## 2. Simulate data

In [ ]:
[gtex_ld]
parameter: genofile = paths
parameter: cwd = path("output")
parameter: job_size = 10
parameter: walltime = "100h"
parameter: mem = "120G"
parameter: numThreads = 2
parameter: B_stoc = 10000
parameter: container = ""
input: genofile, group_by = 1
output: f'{cwd:a}/ld/region_{_index}_ukb_B_stoc_{B_stoc}.rds'
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime, mem = mem, cores = numThreads, tags = f'{step_name}_{_output[0]:bn}'
R: expand = '${ }', stdout = f"{_output:n}.stdout", stderr = f"{_output:n}.stderr", container = container
    library(plink2R)
    library(tidyverse)
    library(MASS)
    library(crayon)
    library(pecotmr)
    library(susieR)
    library(vroom)
    library(pgenlibr)
    
    source("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/code/simulate_linreg.R")
    source("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/code/misc.R")

    # pecotmr GitHub version (conda not yet updated)
    for (f in list.files("/sc/arion/projects/ad-omics/haochen/rss_qc/pecotmr/R/",
                         full.names = TRUE, pattern = "\\.R$")) source(f)

    ## 1. Load genotype (unrelated EUR, trimmed) ------------------------------
    file <- ${_input:r}
    i <- file %>% str_remove("\\.bim$") %>% str_remove(".*/UKBB_")
    UKBB_path <- paste0("/sc/arion/projects/ad-omics/haochen/rss_qc/genotypes/ukbb_unrelated_trimed/result/UKBB_", i)
    UKBB_dat <- plink_process(UKBB_path)
    Xmat <- UKBB_dat$bed
    if (ncol(Xmat) < 500) stop("Too few SNPs — terminating job")

    n_samples <- nrow(Xmat); p <- ncol(Xmat)
    cat(green("Using "), n_samples, green(" samples and "), p, green(" variants.\n"))

    ## 2. Three LD references ------------------------------------------------
    LD_full <- get_correlation(Xmat)

    B_stoc   <- ${B_stoc}
    n_ld_sub <- B_stoc
    idx_ld   <- sample(seq_len(nrow(Xmat)), size = min(n_ld_sub, nrow(Xmat)))
    Xmat_sub  <- Xmat[idx_ld, , drop = FALSE]
    LD_sample <- get_correlation(Xmat_sub)

    W_path <- paste0("/sc/arion/projects/ad-omics/haochen/rss_qc/genotypes/random_vector/result/random_matrix_", B_stoc, ".rds")
    W <- readRDS(W_path)                 # n_samples x B_stoc
    stopifnot(nrow(W) == n_samples)
    U      <- crossprod(Xmat, W)         # p x B_stoc

    ## 3. Save (genotype/LD only — no phenotype, no diffs) -------------------
    ld_data <- list(
      region_id = i,
      LD_full   = LD_full,
      LD_sample = LD_sample,
      U         = U,
      idx_ld    = idx_ld,
      B_stoc    = B_stoc,
      n         = n_samples,
      p         = p,
      mat_dim   = dim(Xmat)
    )
    dir.create(dirname(${_output:r}), recursive = TRUE, showWarnings = FALSE)
    saveRDS(ld_data, ${_output:r})

In [ ]:
[gtex_pheno]
# Phenotype + summary stats — varies with h2g / n_causal. Loads genotype but stores NO LD.
parameter: genofile = paths
parameter: cwd = path("output")
parameter: job_size = 10
parameter: walltime = "100h"
parameter: mem = "120G"
parameter: numThreads = 2
parameter: n_causal = 1
parameter: h2g = 0.05
parameter: total_h2g = True
parameter: container = ""
input: genofile, group_by = 1
output: f'{cwd:a}/pheno/region_{_index}_ncausal_{n_causal}_h2g_{h2g}.rds'
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime, mem = mem, cores = numThreads, tags = f'{step_name}_{_output[0]:bn}'
R: expand = '${ }', stdout = f"{_output:n}.stdout", stderr = f"{_output:n}.stderr", container = container
    library(plink2R)
    library(tidyverse)
    library(MASS)
    library(crayon)
    library(pecotmr)
    library(susieR)
    library(vroom)
    library(pgenlibr)
    
    source("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/code/simulate_linreg.R")
    source("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/code/misc.R")

    # pecotmr GitHub version (conda not yet updated)
    for (f in list.files("/sc/arion/projects/ad-omics/haochen/rss_qc/pecotmr/R/",
                         full.names = TRUE, pattern = "\\.R$")) source(f)

    ## 1. Load genotype (unrelated EUR, trimmed) ------------------------------
    file <- ${_input:r}
    i <- file %>% str_remove("\\.bim$") %>% str_remove(".*/UKBB_")
    UKBB_path <- paste0("/sc/arion/projects/ad-omics/haochen/rss_qc/genotypes/ukbb_unrelated_trimed/result/UKBB_", i)
    UKBB_dat <- plink_process(UKBB_path)
    Xmat <- UKBB_dat$bed
    if (ncol(Xmat) < 500) stop("Too few SNPs — terminating job")

    n_samples <- nrow(Xmat); p <- ncol(Xmat)

    ## 2. Simulate phenotype (uses ALL samples) ------------------------------
    ntrait  <- 1
    ncausal <- ${n_causal}
    h2g     <- ${h2g}

    causal_index <- sample_causal_index(Xmat = UKBB_dat$bed, ncausal = ncausal)
    B_mat <- matrix(0, nrow = p, ncol = ntrait)
    beta  <- sim_beta_fix_variant(G = Xmat, causal_index = causal_index, is_h2g_total = FALSE)
    B_mat[, 1] <- beta

    phenotype <- sim_multi_traits(G = Xmat, B = B_mat, h2g = h2g, is_h2g_total = FALSE)$P

    ## 3. Summary stats ------------------------------------------------------
    target_tb  <- calculate_sumstat(Xmat, as.numeric(phenotype[, 1]))
    sumstat_qc <- target_tb
    z_vec      <- sumstat_qc$z

    ## 4. Save (phenotype + sumstats only; index-matches the LD rds) ----------
    pheno_data <- list(
      region_id  = i,
      sumstat_qc = sumstat_qc,
      z_vec      = z_vec,
      causal_index = causal_index,
      true_vars  = causal_index,
      beta       = beta,
      phenotype  = phenotype,
      n          = n_samples,
      p          = p,
      h2g        = h2g,
      n_causal   = ncausal
    )
    dir.create(dirname(${_output:r}), recursive = TRUE, showWarnings = FALSE)
    saveRDS(pheno_data, ${_output:r})

In [ ]:
# ===== Base config =====
base_dir="/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/"
job_name="UKBB_sketch_dimension"
job_dir="${base_dir}/${job_name}"

# ===== Create structure =====
mkdir -p "${job_dir}"/{code,result,log}
cd "${job_dir}/code"

work_dir="${job_dir}"

# ============================================================
# PART 1 — LD / genotype sketch  (loop over B_stoc only)
# ============================================================
cat << 'EOF' > ${job_name}_ld.sh
#!/bin/bash -l
#BSUB -P acc_ad-omics
#BSUB -q premium
#BSUB -J JOB_NAME_ld_B_STOC
#BSUB -n 3
#BSUB -R span[hosts=1]
#BSUB -R rusage[mem=80000]
#BSUB -W 72:00
#BSUB -o /WORK_DIR/log/JOB_NAME_ld_B_STOC.%J.out
#BSUB -e /WORK_DIR/log/JOB_NAME_ld_B_STOC.%J.err

export PATH=/sc/arion/projects/ad-omics/haochen/env/R_sc/bin:$PATH:/usr/bin:/bin

sos run "/sc/arion/projects/ad-omics/haochen/rss_qc/2026_Simulation_Sinai/04.Fig2D_S3A.Sketch_Dimension_LD.ipynb" gtex_ld \
    --genofile $(echo /sc/arion/projects/ad-omics/haochen/rss_qc/genotypes/ukbb_unrelated_trimed/result/*.bim) \
    --B_stoc B_STOC \
    --cwd WORK_DIR/result/B_B_STOC/ -j 5
EOF

chmod +x ${job_name}_ld.sh

for B_stoc in 500 1000 2000 3000 5000 10000 20000; do
    output_script="${job_name}_ld_B_stoc_${B_stoc}.sh"
    sed -e "s|B_STOC|${B_stoc}|g" \
        -e "s|WORK_DIR|${work_dir}|g" \
        -e "s|JOB_NAME|${job_name}|g" \
      "${job_name}_ld.sh" > "${output_script}"

    sb "${output_script}"
done

# ============================================================
# PART 2 — Phenotype + sumstats  (loop over causal x h2g only)
# ============================================================
cat << 'EOF' > ${job_name}_pheno.sh
#!/bin/bash -l
#BSUB -P acc_ad-omics
#BSUB -q premium
#BSUB -J JOB_NAME_ph_c_CAUSAL_h_H2G
#BSUB -n 3
#BSUB -R span[hosts=1]
#BSUB -R rusage[mem=80000]
#BSUB -W 72:00
#BSUB -o /WORK_DIR/log/JOB_NAME_ph_c_CAUSAL_h_H2G.%J.out
#BSUB -e /WORK_DIR/log/JOB_NAME_ph_c_CAUSAL_h_H2G.%J.err

export PATH=/sc/arion/projects/ad-omics/haochen/env/R_sc/bin:$PATH:/usr/bin:/bin

sos run "/sc/arion/projects/ad-omics/haochen/rss_qc/2026_Simulation_Sinai/04.Fig2D_S3A.Sketch_Dimension_LD.ipynb" gtex_pheno \
    --genofile $(echo /sc/arion/projects/ad-omics/haochen/rss_qc/genotypes/ukbb_unrelated_trimed/result/*.bim) \
    --n_causal CAUSAL --h2g H2G \
    --cwd WORK_DIR/result/h2g_H2G/causal_CAUSAL/ -j 5
EOF

chmod +x ${job_name}_pheno.sh

# NCP-matched h2g
for causal in 1 2 3; do
  for h2g in 0.0006 0.0008; do
      output_script="${job_name}_pheno_causal_${causal}_h2g_${h2g}.sh"
      sed -e "s|CAUSAL|${causal}|g" \
          -e "s|WORK_DIR|${work_dir}|g" \
          -e "s|H2G|${h2g}|g" \
          -e "s|JOB_NAME|${job_name}|g" \
        "${job_name}_pheno.sh" > "${output_script}"

       sb "${output_script}"
  done
done

In [ ]:
[LD_mismatch]
parameter: resfile = paths
# All LD .rds files across B settings; combined into ONE output table.
parameter: cwd = path("output")
parameter: job_size = 10
parameter: walltime = "100h"
parameter: mem = "72G"
parameter: numThreads = 3
parameter: container = ""
# ---- Metric knobs ------------------------------------------------------------
parameter: ld_thresh = 0.05    # keep off-diagonal pairs with |R_insample| > this
parameter: min_pairs = 10      # skip references with fewer kept pairs
input: resfile
output: f'{cwd:a}/ld_mismatch_combined.rds', f'{cwd:a}/ld_mismatch_combined.csv'
task: trunk_workers = 1, walltime = walltime, mem = mem, cores = numThreads,
      tags = f'{step_name}_{_output[0]:bn}'
R:  expand = '${ }', stdout = f"{_output[0]:n}.stdout", stderr = f"{_output[0]:n}.stderr", container = container

    suppressMessages(library(parallel))
    `%||%` <- function(a, b) if (is.null(a)) b else a
    files <- c(${_input:r,})

    cov2cor_safe <- function(M) {
      d <- diag(M); if (any(!is.finite(d)) || any(d <= 0)) return(M); cov2cor(M)
    }

    ## per-region worker: 2-row df (sketch/subset) vs in-sample, or NULL on error
    process_one <- function(f) {
      tryCatch({
        res     <- readRDS(f)
        R_ins   <- cov2cor_safe(res$LD_full)
        p       <- nrow(R_ins)

        # sketch: U is p x B  ->  R_hat = cov2cor(U U^T / B)
        U <- res$U; if (nrow(U) != p) U <- t(U)
        R_sketch <- cov2cor(tcrossprod(U) / ncol(U))
        # subset: dense LD from n_sub individuals
        R_sub    <- cov2cor_safe(res$LD_sample)

        metric <- function(R_ref, B) {
          ut  <- upper.tri(R_ins)
          ins <- R_ins[ut]; ref <- R_ref[ut]
          ok  <- is.finite(ins) & is.finite(ref) & (abs(ins) > ${ld_thresh})
          if (sum(ok) < ${min_pairs})
            return(c(mse = NA_real_, med_sq = NA_real_, max_abs_diff = NA_real_,
                     concord = NA_real_, D2 = NA_real_, n_pairs = sum(ok)))
          d <- ref[ok] - ins[ok]
          c(mse          = mean(d^2),
            med_sq       = median(d^2),
            max_abs_diff = max(abs(d)),
            concord      = suppressWarnings(cor(ref[ok], ins[ok])),
            D2           = if (is.finite(B)) mean(d^2) - mean((1 + ref[ok]^2) / B) else NA_real_,
            n_pairs      = sum(ok))
        }

        B_sketch <- ncol(U)
        H <- list(sketch = list(R = R_sketch, B = B_sketch),
                  subset = list(R = R_sub,    B = NA_real_))   # set B = length(res$idx_ld) to enable subset D2

        m <- t(sapply(H, function(h) metric(h$R, h$B)))
        data.frame(file = basename(f), reference = names(H), m,
                   p = p, B = sapply(H, `[[`, "B"),
                   B_setting = res$B_stoc %||% B_sketch,
                   row.names = NULL, check.names = FALSE)
      }, error = function(e) { message("FAILED: ", basename(f), " : ", conditionMessage(e)); NULL })
    }

    parts <- mclapply(files, process_one, mc.cores = ${numThreads})
    ok    <- !vapply(parts, is.null, logical(1))
    dat   <- do.call(rbind, parts[ok])

    saveRDS(dat, ${_output[0]:r})
    write.csv(dat, ${_output[1]:r}, row.names = FALSE)
    cat(sprintf("combined %d/%d regions -> %d rows\n", sum(ok), length(files), nrow(dat)))

In [ ]:
# ===== Base config =====
base_dir="/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/"
job_name="UKBB_LD_mismatch"
job_dir="${base_dir}/${job_name}"

mkdir -p "${job_dir}"/{code,log}
cd "${job_dir}/code"
work_dir="${job_dir}"

cat << EOF > ${job_name}.sh
#!/bin/bash -l
#BSUB -P acc_ad-omics
#BSUB -q premium
#BSUB -J ${job_name}
#BSUB -n 3
#BSUB -R span[hosts=1]
#BSUB -R rusage[mem=24000]
#BSUB -W 100:00
#BSUB -o ${work_dir}/log/${job_name}.%J.out
#BSUB -e ${work_dir}/log/${job_name}.%J.err

export PATH=/sc/arion/projects/ad-omics/haochen/env/R_sc/bin:\$PATH:/usr/bin:/bin

sos run "/sc/arion/projects/ad-omics/haochen/rss_qc/2026_Simulation_Sinai/04.Fig2D_S3A.Sketch_Dimension_LD.ipynb" LD_mismatch \\
    --resfile \$(echo /sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/UKBB_sketch_dimension/result/B_*/ld/*.rds) \\
    --cwd     /sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/UKBB_sketch_dimension/ld_mismatch
EOF

chmod +x ${job_name}.sh
sb ${job_name}.sh